In [ ]:
import os
# Prevent memory fragmentation before imports
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import random
import numpy as np
import torch
import torch.nn as nn
import gc
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # Corrected import for modern PyTorch
from torch.cuda.amp import autocast, GradScaler  # For Mixed Precision
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
from google.colab import drive
import logging

# Clear any existing memory from previous runs
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()

clear_gpu_memory()

In [ ]:
# ==========================================
# 2. GLOBAL CONFIGURATION
# ==========================================
CONFIG = {
    "model_name": "microsoft/deberta-v3-large",
    "max_length": 512,
    "batch_size": 2,              # Smallest possible for DeBERTa-Large
    "gradient_accumulation": 4,   # Effective batch size = 4
    "epochs": 5,
    "learning_rate": 1e-5,
    "seed": 42,
    "downsample_ratio": 0.3,      # Keep 30% of 'no_relation' samples
    "unanswerable_weight": 0.5     # Penalty for no_relation in loss
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])

# Path Setup
BASE_PATH = "/input/data_dialogre"
SAVE_DIR = "/kaggle/output"
os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
# ==========================================
# 3. DATA PROCESSING & NEGATIVE SAMPLING
# ==========================================
SOCIAL_RELATIONS = [
    "per:friends", "per:spouse", "per:children", "per:parents", 
    "per:siblings", "per:other_family", "per:girl/boyfriend",
    "per:positive_impression", "per:negative_impression", "per:acquaintance"
]

selected_labels = ["unanswerable"] + SOCIAL_RELATIONS
label2id = {label: i for i, label in enumerate(selected_labels)}
id2label = {i: label for label, i in label2id.items()}

SPECIAL_TOKENS = {'s_start': '[SS]', 's_end': '[/SS]', 'o_start': '[OS]', 'o_end': '[/OS]'}

def load_and_process_data(file_path, tokenizer, is_train=True):
    with open(file_path, 'r') as f:
        raw_data = json.load(f)
    
    processed = []
    for conv in raw_data:
        dialogue_text = " ".join(conv[0])
        for rel in conv[1]:
            r_type = rel['r'][0]
            x, y = rel['x'], rel['y']
            
            # Label Filtering & Negative Sampling
            is_social = r_type in SOCIAL_RELATIONS
            is_none = r_type in ["unanswerable"]
            
            if is_social or is_none:
                # Apply Downsampling to no_relation ONLY during training
                if is_train and is_none and random.random() > CONFIG['downsample_ratio']:
                    continue
                
                label = label2id[r_type] if is_social else label2id["unanswerable"]
                
                # Mark Entities
                marked_text = dialogue_text.replace(x, f"{SPECIAL_TOKENS['s_start']} {x} {SPECIAL_TOKENS['s_end']}", 1)
                marked_text = marked_text.replace(y, f"{SPECIAL_TOKENS['o_start']} {y} {SPECIAL_TOKENS['o_end']}", 1)
                
                processed.append({'text': marked_text, 'label': label})
    return processed

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
tokenizer.add_special_tokens({'additional_special_tokens': list(SPECIAL_TOKENS.values())})

train_data = load_and_process_data(os.path.join(BASE_PATH, "train.json"), tokenizer, is_train=True)
val_data = load_and_process_data(os.path.join(BASE_PATH, "dev.json"), tokenizer, is_train=False)

In [ ]:
# ==========================================
# 4. DATASET & MODEL (MEMORY OPTIMIZED)
# ==========================================
class RelationDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self): return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        encoding = self.tokenizer(item['text'], max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'labels': torch.tensor(item['label'], dtype=torch.long)}

class REModel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        
        # 1. Gradient Checkpointing (Saves ~50% VRAM)
        self.encoder.gradient_checkpointing_enable() 
        
        self.encoder.resize_token_embeddings(len(tokenizer))
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.classifier(outputs.last_hidden_state[:, 0, :])
        
        loss = None
        if labels is not None:
            # 2. Weighted Loss for Class Imbalance
            weights = torch.ones(len(label2id)).to(input_ids.device)
            weights[0] = CONFIG['unanswerable_weight']
            loss_fct = nn.CrossEntropyLoss(weight=weights)
            loss = loss_fct(logits, labels)
            
        return loss, logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = REModel(CONFIG['model_name'], len(label2id)).to(device)


In [ ]:
# ==========================================
# 5. TRAINING WITH AMP & ACCUMULATION (
# ==========================================

train_loader = DataLoader(RelationDataset(train_data, tokenizer, CONFIG['max_length']), batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(RelationDataset(val_data, tokenizer, CONFIG['max_length']), batch_size=CONFIG['batch_size'])

optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'])
scaler = GradScaler() # For Mixed Precision
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * CONFIG['epochs'])

def train_epoch(model, loader):
    model.train()
    total_loss = 0
    
    # 1. Reset gradients at start of epoch to prevent summation from previous runs
    optimizer.zero_grad() 
    
    for i, batch in enumerate(loader):
        # 2. Mixed Precision Forward Pass
        with autocast():
            loss, _ = model(
                batch['input_ids'].to(device), 
                batch['attention_mask'].to(device), 
                batch['labels'].to(device)
            )
            # Normalize loss based on accumulation steps
            loss = loss / CONFIG['gradient_accumulation']
        
        # 3. Backpropagation with Scaler
        scaler.scale(loss).backward()
        
        # 4. Gradient Accumulation Step + Edge Case Handling
        # Checks if we reached accumulation limit OR the very last batch of the epoch
        if (i + 1) % CONFIG['gradient_accumulation'] == 0 or (i + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            
            # 5. Efficient Memory Flow: Zeroing immediately after update
            # Use set_to_none=True for slightly better performance on Large models
            optimizer.zero_grad(set_to_none=True) 
            scheduler.step()
            
        total_loss += loss.item()
        
    return total_loss / len(loader)

# --- Standard Evaluation ---
def evaluate(model, loader):
    model.eval()
    true, preds = [], []
    with torch.no_grad():
        for batch in loader:
            _, logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            true.extend(batch['labels'].cpu().numpy())
    return true, preds

# --- Execute Training ---
best_f1 = 0
for epoch in range(CONFIG['epochs']):
    print(f"Epoch {epoch+1}")
    train_loss = train_epoch(model, train_loader)
    true, pred = evaluate(model, val_loader)
    
    # Calculating metrics
    f1 = f1_score(true, pred, average='macro')
    
    print(f"Loss: {train_loss:.4f} | Val F1: {f1:.4f}")
    
    # Save the best model weights
    if f1 > best_f1:
        best_f1 = f1
        save_file = os.path.join(SAVE_DIR, "deberta-v3_model.pth")
        torch.save(model.state_dict(), save_file)
        print(f"✓ New best model saved to: {save_file}")